# Find Vulnerable Libraries

In [2]:
# Data

# Libraries snippet
# path: /Users/dmk6603/Documents/swdb_opensource/5-generate_sbom/libraries.csv
# project,name,version,type,purl,cpe,language,pkg_type,path
# APACHE-Hive,DummyUDF,UNKNOWN,library,pkg:maven/DummyUDF/DummyUDF,cpe:2.3:a:DummyUDF:DummyUDF:*:*:*:*:*:*:*:*,java,java-archive,/itests/hive-unit/testUdf/DummyUDF.jar
# APACHE-Hive,HikariCP,4.0.3,library,pkg:maven/com.zaxxer/HikariCP@4.0.3,cpe:2.3:a:com.zaxxer:HikariCP:4.0.3:*:*:*:*:*:*:*,java,java-archive,/standalone-metastore/metastore-common/pom.xml

# Vulnerabilities snippet
# path: /Users/dmk6603/Documents/swdb_opensource/5.1-grype_vulnerabilities/vulnerabilities.csv
# snippet
# project,vuln_id,cve,severity,risk_score,epss,epss_percentile,cwe,cvss_score,cvss_vector,description,fix_state,fix_versions,pkg_name,pkg_version,pkg_type,pkg_language,purl,data_source
# APACHE-Hive,GHSA-cqqj-4p63-rrmm,CVE-2019-20444,Critical,10.777645000000001,0.11909,0.93698,CWE-444,9.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:N,HTTP Request Smuggling in Netty,not-fixed,,netty,3.10.5.Final,java-archive,java,pkg:maven/io.netty/netty@3.10.5.Final,https://github.com/advisories/GHSA-cqqj-4p63-rrmm
# APACHE-Hive,GHSA-8vhq-qq4p-grq3,CVE-2017-1000487,Critical,7.33012,0.07798,0.91928,CWE-78,9.8,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H,OS Command Injection in Plexus-utils,fixed,3.0.16,plexus-utils,1.5.6,java-archive,java,pkg:maven/org.codehaus.plexus/plexus-utils@1.5.6,https://github.com/advisories/GHSA-8vhq-qq4p-grq3


In [ ]:
import pandas as pd

# Load libraries data
libraries_df = pd.read_csv('/Users/dmk6603/Documents/swdb_opensource/5-generate_sbom/libraries.csv')

# Load vulnerabilities data
vulnerabilities_df = pd.read_csv('/Users/dmk6603/Documents/swdb_opensource/5.1-grype_vulnerabilities/vulnerabilities.csv')

# Count vulnerabilities per (project, package name, version)
vuln_counts = (
    vulnerabilities_df
    .groupby(['project', 'pkg_name', 'pkg_version'])
    .size()
    .reset_index(name='vuln_count')
)

# Merge libraries with vuln counts on project + name/pkg_name + version/pkg_version
libraries_df = libraries_df.merge(
    vuln_counts,
    left_on=['project', 'name', 'version'],
    right_on=['project', 'pkg_name', 'pkg_version'],
    how='left'
)

# Clean up the extra columns from the right side of the merge
libraries_df = libraries_df.drop(columns=['pkg_name', 'pkg_version'])

# Fill NaN vuln_count with 0 (libraries with no vulnerabilities)
libraries_df['vuln_count'] = libraries_df['vuln_count'].fillna(0).astype(int)

# Add boolean vulnerable column
libraries_df['vulnerable'] = libraries_df['vuln_count'] > 0

print(libraries_df[['project', 'name', 'version', 'vulnerable', 'vuln_count']].head(20))
print(f"\nTotal libraries: {len(libraries_df)}")
print(f"Vulnerable libraries: {libraries_df['vulnerable'].sum()}")

# save in a new file
libraries_df.to_csv('/Users/dmk6603/Documents/swdb_opensource/7-vulnerable_libraries/libraries_with_vulns.csv', index=False)


        project                  name           version  vulnerable  vuln_count
0   APACHE-Hive              DummyUDF           UNKNOWN       False           0
1   APACHE-Hive              HikariCP             4.0.3       False           0
2   APACHE-Hive         RoaringBitmap             1.3.0       False           0
3   APACHE-Hive                   ST4             4.0.4       False           0
4   APACHE-Hive         accumulo-core            1.10.4       False           0
5   APACHE-Hive         accumulo-fate            1.10.4       False           0
6   APACHE-Hive  accumulo-minicluster            1.10.4       False           0
7   APACHE-Hive        accumulo-start            1.10.4       False           0
8   APACHE-Hive        accumulo-trace            1.10.4       False           0
9   APACHE-Hive      actions/checkout                v2       False           0
10  APACHE-Hive      actions/checkout                v3       False           0
11  APACHE-Hive    actions/setup-java   

# How many products single library affects?

## Splice and create new column with project name.

In [13]:
# create new column in metrics file: product and vendor
# replace _ with space in product and vendor
# replace space at the end of vendor with letter .
# replace _ with space in product and vendor

libraries_df['vendor'] = libraries_df['project'].apply(lambda x: x.split('-')[0].replace('_', ' '))
# replace space at the end of vendor with letter .
libraries_df['vendor'] = libraries_df['vendor'].apply(lambda x: x[:-1] + '.' if x.endswith(' ') else x)
libraries_df['product'] = libraries_df['project'].apply(lambda x: x.split('-')[1].replace('_', ' '))

# save in a file
libraries_df.to_csv('/Users/dmk6603/Documents/swdb_opensource/7-vulnerable_libraries/libraries_with_vulns.csv', index=False)
# head
print(libraries_df.head(20))

        project                  name           version     type                                               purl                                                cpe language       pkg_type                                               path  vuln_count  vulnerable product  vendor
0   APACHE-Hive              DummyUDF           UNKNOWN  library                        pkg:maven/DummyUDF/DummyUDF        cpe:2.3:a:DummyUDF:DummyUDF:*:*:*:*:*:*:*:*     java   java-archive             /itests/hive-unit/testUdf/DummyUDF.jar           0       False    Hive  APACHE
1   APACHE-Hive              HikariCP             4.0.3  library                pkg:maven/com.zaxxer/HikariCP@4.0.3  cpe:2.3:a:com.zaxxer:HikariCP:4.0.3:*:*:*:*:*:*:*     java   java-archive     /standalone-metastore/metastore-common/pom.xml           0       False    Hive  APACHE
2   APACHE-Hive         RoaringBitmap             1.3.0  library    pkg:maven/org.roaringbitmap/RoaringBitmap@1.3.0  cpe:2.3:a:org.roaringbitmap:RoaringBi

# Counts of Vulnerable Libraries Propagation

In [14]:
# path: /Users/dmk6603/Documents/swdb_opensource/7-vulnerable_libraries/libraries_with_vulns.csv
# snippet
# project,name,version,type,purl,cpe,language,pkg_type,path,vuln_count,vulnerable,product,vendor
# APACHE-Hive,DummyUDF,UNKNOWN,library,pkg:maven/DummyUDF/DummyUDF,cpe:2.3:a:DummyUDF:DummyUDF:*:*:*:*:*:*:*:*,java,java-archive,/itests/hive-unit/testUdf/DummyUDF.jar,0,False,Hive,APACHE
# APACHE-Hive,HikariCP,4.0.3,library,pkg:maven/com.zaxxer/HikariCP@4.0.3,cpe:2.3:a:com.zaxxer:HikariCP:4.0.3:*:*:*:*:*:*:*,java,java-archive,/standalone-metastore/metastore-common/pom.xml,0,False,Hive,APACHE
# APACHE-Hive,RoaringBitmap,1.3.0,library,pkg:maven/org.roaringbitmap/RoaringBitmap@1.3.0,cpe:2.3:a:org.roaringbitmap:RoaringBitmap:1.3.0:*:*:*:*:*:*:*,java,java-archive,/itests/qtest-iceberg/pom.xml,0,False,Hive,APACHE

# path:/Users/dmk6603/Documents/swdb_opensource/1-indentify_open_source/data/swdb_universe_installs.csv 
# snippet
# VendorName,Product,ProductId,TabKeyNewId,ProductSeries,ProductCategory,Total Sites,US Sites,Total Enterprises,Enterprises in US
# AppNexus,AppNexus,4611,4907,Advertising,Ad Exchanges,"696,761","471,708","257,895","130,146"
# AppNexus,AppNexus,4611,4907,Advertising,Ad Exchanges,"696,761","471,708","257,895","130,146"



In [16]:
"""
Build: Vulnerable Library Impact Table
=======================================
Output columns:
  - Vulnerable Library      : name + version
  - CVE ID                  : the actual CVE identifier (e.g. CVE-2019-20444)
  - CVE Severity            : Critical / High / Medium / Low
  - Products Affected       : count of distinct products that ship this library
  - Affected Products       : names of those products (comma-separated)
  - Enterprises Exposed     : sum of Total Enterprises across those products

One row per (library name + version + CVE).

Merge keys:
  libraries  → installs     : vendor == VendorName  AND  product == Product
  libraries  → vulns        : project + name + version == project + pkg_name + pkg_version
"""

import pandas as pd

# ── 0. File paths ─────────────────────────────────────────────────────────────
LIBRARIES_PATH = "/Users/dmk6603/Documents/swdb_opensource/7-vulnerable_libraries/libraries_with_vulns.csv"
VULNS_PATH     = "/Users/dmk6603/Documents/swdb_opensource/5.1-grype_vulnerabilities/vulnerabilities.csv"
INSTALLS_PATH  = "/Users/dmk6603/Documents/swdb_opensource/1-indentify_open_source/data/swdb_universe_installs.csv"
OUTPUT_PATH    = "/Users/dmk6603/Documents/swdb_opensource/vulnerable_library_impact.csv"

# ── 1. Load ───────────────────────────────────────────────────────────────────
print("Loading files...")
libs     = pd.read_csv(LIBRARIES_PATH)
vulns    = pd.read_csv(VULNS_PATH)
installs = pd.read_csv(INSTALLS_PATH)

# Strip whitespace from all column names
for df in [libs, vulns, installs]:
    df.columns = df.columns.str.strip()

print(f"  Libraries rows   : {len(libs):,}")
print(f"  Vulnerability rows: {len(vulns):,}")
print(f"  Install rows     : {len(installs):,}")

# ── 2. Keep only vulnerable libraries ────────────────────────────────────────
vuln_libs = libs[libs["vulnerable"] == True].copy()
print(f"\nVulnerable library rows: {len(vuln_libs):,}")
print(f"Distinct libraries      : {vuln_libs[['name','version']].drop_duplicates().shape[0]:,}")
print(f"Distinct products       : {vuln_libs[['vendor','product']].drop_duplicates().shape[0]:,}")

# ── 3. Clean installs: parse Total Enterprises, de-duplicate ─────────────────
installs["Total Enterprises"] = (
    installs["Total Enterprises"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype(int)
)

# A (VendorName, Product) pair can appear multiple times — keep MAX
installs_dedup = (
    installs
    .groupby(["VendorName", "Product"], as_index=False)["Total Enterprises"]
    .max()
)
print(f"\nInstalls unique (vendor, product) pairs: {len(installs_dedup):,}")

# ── 4. Build a clean product → enterprise map ─────────────────────────────────
# Merge vuln_libs with installs to know how many enterprises each product has
libs_with_ent = vuln_libs.merge(
    installs_dedup,
    left_on=["vendor", "product"],
    right_on=["VendorName", "Product"],
    how="left"
)

unmatched = libs_with_ent["Total Enterprises"].isna().sum()
if unmatched > 0:
    print(f"\n⚠  {unmatched:,} library rows had no matching installs entry (will count as 0 enterprises).")
    missing = (
        libs_with_ent[libs_with_ent["Total Enterprises"].isna()]
        [["vendor", "product"]].drop_duplicates()
    )
    print("   Unmatched (vendor, product):")
    print(missing.to_string(index=False))

libs_with_ent["Total Enterprises"] = libs_with_ent["Total Enterprises"].fillna(0).astype(int)

# ── 5. Prepare vulnerabilities: one row per (project, pkg_name, pkg_version, CVE) ──
# Keep only the columns we need; drop duplicate (project, pkg, version, cve) combos
vulns_clean = (
    vulns[["project", "pkg_name", "pkg_version", "cve", "severity"]]
    .drop_duplicates()
    .copy()
)

# Some CVE fields may be blank — flag them
no_cve = vulns_clean["cve"].isna().sum()
if no_cve > 0:
    print(f"\n⚠  {no_cve:,} vulnerability rows have no CVE ID — they will show as 'N/A'.")
vulns_clean["cve"] = vulns_clean["cve"].fillna("N/A")
vulns_clean["severity"] = vulns_clean["severity"].fillna("Unknown")

print(f"\nDistinct (library, CVE) pairs in vulns file: {len(vulns_clean):,}")
print("Severity distribution in vulns file:")
print(vulns_clean["severity"].value_counts().to_string())

# ── 6. Merge libraries → vulnerabilities ─────────────────────────────────────
# Result: one row per (library instance in a product) x CVE
libs_with_cve = libs_with_ent.merge(
    vulns_clean,
    left_on=["project", "name", "version"],
    right_on=["project", "pkg_name", "pkg_version"],
    how="inner"   # inner: only rows where library actually has a CVE
)

print(f"\nRows after library x CVE merge: {len(libs_with_cve):,}")

# ── 7. Aggregate: one row per (library name + version + CVE) ─────────────────
# For each group: collect distinct products and sum their enterprises
def aggregate_group(group):
    # De-duplicate at (vendor, product) level — a library can appear in
    # multiple projects of the same product, but we count the product once
    product_level = (
        group[["vendor", "product", "Total Enterprises"]]
        .drop_duplicates(subset=["vendor", "product"])
    )
    products_count      = product_level["product"].nunique()
    products_list       = ", ".join(sorted(product_level["product"].dropna().unique()))
    enterprises_exposed = product_level["Total Enterprises"].sum()
    # severity is the same for all rows in this group (same CVE)
    severity = group["severity"].iloc[0]

    return pd.Series({
        "CVE Severity"       : severity,
        "Products Affected"  : products_count,
        "Affected Products"  : products_list,
        "Enterprises Exposed": enterprises_exposed,
    })

print("\nAggregating impact per library x CVE...")
impact = (
    libs_with_cve
    .groupby(["name", "version", "cve"], dropna=False)
    .apply(aggregate_group)
    .reset_index()
)

# ── 8. Build and sort final table ─────────────────────────────────────────────
impact["Vulnerable Library"] = impact["name"] + " " + impact["version"].astype(str)
impact["CVE ID"]             = impact["cve"]

# Sort: severity order, then enterprises exposed descending
severity_rank = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3, "Unknown": 4}
impact["_rank"] = impact["CVE Severity"].map(severity_rank).fillna(99)
impact = impact.sort_values(["_rank", "Enterprises Exposed"], ascending=[True, False])

final = impact[[
    "Vulnerable Library",
    "CVE ID",
    "CVE Severity",
    "Products Affected",
    "Affected Products",
    "Enterprises Exposed",
]].reset_index(drop=True)

# ── 9. Save ───────────────────────────────────────────────────────────────────
final.to_csv(OUTPUT_PATH, index=False)

print(f"\n✅ Done! Saved to: {OUTPUT_PATH}")
print(f"   Total rows       : {len(final):,}")
print(f"   Distinct CVEs    : {final['CVE ID'].nunique():,}")
print(f"   Distinct libs    : {final['Vulnerable Library'].nunique():,}")
print(f"\nTop 15 rows (sorted by severity then enterprises):")
print(final.head(15).to_string(index=False))

Loading files...
  Libraries rows   : 209,552
  Vulnerability rows: 11,302
  Install rows     : 21,457

Vulnerable library rows: 6,477
Distinct libraries      : 2,187
Distinct products       : 164

Installs unique (vendor, product) pairs: 10,718

⚠  56 library rows had no matching installs entry (will count as 0 enterprises).
   Unmatched (vendor, product):
                                 vendor                                   product
              CKSource sp  z o o  sp k.                                  CKEditor
                          DeNA Co  Ltd.                                       H2O
Ethereum Foundation  Stiftung Ethereum.                                  Ethereum
                                 H2O ai                                    H20 ai
                                Node js                                   Node js
                                Red Hat JBoss Enterprise Application Server  EAS 
             The F  Software Foundation                           

/var/folders/fj/wtzx880x4v7g0q54zrwpf4gc0000gr/T/ipykernel_92332/1197759284.py:143: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(aggregate_group)


# Count of ONLY Libraries propagation
## We are not considering only vulnerable libraries, but all libraries

In [17]:
"""
Build: General Library Risk Overview
======================================
All libraries (vulnerable and not), showing adoption breadth and risk metrics.

Output columns:
  - Library Name
  - Version
  - Total Products Using It     : distinct products that ship this library
  - Affected Products           : comma-separated product names
  - Total Enterprises Exposed   : sum of enterprises across those products (de-duped)
  - Total CVEs                  : total CVE count for this library+version
  - Worst Severity              : highest severity across all CVEs (Critical > High > Medium > Low)
  - # Critical                  : count of Critical CVEs
  - # High                      : count of High CVEs
  - # Medium                    : count of Medium CVEs
  - # Low                       : count of Low CVEs
  - Fix Available               : Yes / Partial / No / — (if no CVEs)

Merge keys:
  libraries → installs : vendor == VendorName  AND  product == Product
  libraries → vulns    : project + name + version == project + pkg_name + pkg_version
"""

import pandas as pd
import numpy as np

# ── 0. File paths ─────────────────────────────────────────────────────────────
LIBRARIES_PATH = "/Users/dmk6603/Documents/swdb_opensource/7-vulnerable_libraries/libraries_with_vulns.csv"
VULNS_PATH     = "/Users/dmk6603/Documents/swdb_opensource/5.1-grype_vulnerabilities/vulnerabilities.csv"
INSTALLS_PATH  = "/Users/dmk6603/Documents/swdb_opensource/1-indentify_open_source/data/swdb_universe_installs.csv"
OUTPUT_PATH    = "/Users/dmk6603/Documents/swdb_opensource/general_library_risk.csv"

# ── 1. Load ───────────────────────────────────────────────────────────────────
print("Loading files...")
libs     = pd.read_csv(LIBRARIES_PATH)
vulns    = pd.read_csv(VULNS_PATH)
installs = pd.read_csv(INSTALLS_PATH)

for df in [libs, vulns, installs]:
    df.columns = df.columns.str.strip()

print(f"  Libraries rows    : {len(libs):,}")
print(f"  Vulnerability rows: {len(vulns):,}")
print(f"  Install rows      : {len(installs):,}")

# ── 2. Clean installs: parse Total Enterprises, de-duplicate ─────────────────
installs["Total Enterprises"] = (
    installs["Total Enterprises"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype(int)
)

# Keep MAX enterprises per (VendorName, Product) — guards against duplicate rows
installs_dedup = (
    installs
    .groupby(["VendorName", "Product"], as_index=False)["Total Enterprises"]
    .max()
)
print(f"\nInstalls unique (vendor, product) pairs: {len(installs_dedup):,}")

# ── 3. Merge ALL libraries → installs ────────────────────────────────────────
# We keep ALL libraries (not just vulnerable ones)
libs_with_ent = libs.merge(
    installs_dedup,
    left_on=["vendor", "product"],
    right_on=["VendorName", "Product"],
    how="left"
)

unmatched = libs_with_ent["Total Enterprises"].isna().sum()
if unmatched > 0:
    print(f"\n⚠  {unmatched:,} library rows had no matching installs entry (will count as 0 enterprises).")

libs_with_ent["Total Enterprises"] = libs_with_ent["Total Enterprises"].fillna(0).astype(int)

# ── 4. Prepare vulnerabilities ────────────────────────────────────────────────
vulns_clean = vulns[["project", "pkg_name", "pkg_version", "cve", "severity", "fix_state", "fix_versions"]].copy()
vulns_clean["cve"]      = vulns_clean["cve"].fillna("N/A")
vulns_clean["severity"] = vulns_clean["severity"].fillna("Unknown")

# De-duplicate: one row per (project, pkg_name, pkg_version, cve)
vulns_clean = vulns_clean.drop_duplicates(subset=["project", "pkg_name", "pkg_version", "cve"])

print(f"\nDistinct (library, CVE) pairs: {len(vulns_clean):,}")
print("Severity distribution:")
print(vulns_clean["severity"].value_counts().to_string())

# ── 5. Build adoption stats per (name, version) ───────────────────────────────
# For each library+version: distinct products and total enterprises
# De-dup at (vendor, product) level before aggregating
print("\nBuilding adoption stats...")

def adoption_agg(group):
    product_level = (
        group[["vendor", "product", "Total Enterprises"]]
        .drop_duplicates(subset=["vendor", "product"])
    )
    return pd.Series({
        "Total Products Using It"   : product_level["product"].nunique(),
        "Affected Products"         : ", ".join(sorted(product_level["product"].dropna().unique())),
        "Total Enterprises Exposed" : product_level["Total Enterprises"].sum(),
    })

adoption = (
    libs_with_ent
    .groupby(["name", "version"], dropna=False)
    .apply(adoption_agg)
    .reset_index()
)

print(f"Distinct (library, version) entries: {len(adoption):,}")

# ── 6. Build risk stats per (name, version) ───────────────────────────────────
# Merge libs → vulns then aggregate CVE metrics
# Use all rows of libs (not just vuln_libs) to match via project
libs_for_risk = libs[["project", "name", "version"]].drop_duplicates()

libs_with_vulns = libs_for_risk.merge(
    vulns_clean,
    left_on=["project", "name", "version"],
    right_on=["project", "pkg_name", "pkg_version"],
    how="left"
)

# Severity ordering for "worst severity"
SEV_ORDER = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3, "Unknown": 4}

def worst_severity(severities):
    severities = severities.dropna()
    if len(severities) == 0:
        return "—"
    ranked = severities.map(lambda s: SEV_ORDER.get(s, 99))
    return severities.iloc[ranked.argmin()]

def fix_status(fix_states):
    """
    fixed       → all CVEs have a fix
    not-fixed   → none have a fix
    Partial     → some have a fix
    —           → no CVEs at all
    """
    fix_states = fix_states.dropna()
    if len(fix_states) == 0:
        return "—"
    has_fix    = (fix_states.str.lower() == "fixed").sum()
    has_no_fix = (fix_states.str.lower() == "not-fixed").sum()
    if has_fix > 0 and has_no_fix > 0:
        return "Partial"
    elif has_fix > 0:
        return "Yes"
    else:
        return "No"

print("Building risk stats...")

risk = (
    libs_with_vulns
    .groupby(["name", "version"], dropna=False)
    .agg(
        Total_CVEs      = ("cve",       lambda x: x.dropna().nunique()),
        Worst_Severity  = ("severity",  worst_severity),
        Cnt_Critical    = ("severity",  lambda x: (x == "Critical").sum()),
        Cnt_High        = ("severity",  lambda x: (x == "High").sum()),
        Cnt_Medium      = ("severity",  lambda x: (x == "Medium").sum()),
        Cnt_Low         = ("severity",  lambda x: (x == "Low").sum()),
        Fix_Available   = ("fix_state", fix_status),
    )
    .reset_index()
)

# Libraries with 0 CVEs: fix the worst severity display
risk.loc[risk["Total_CVEs"] == 0, "Worst_Severity"] = "—"
risk.loc[risk["Total_CVEs"] == 0, "Fix_Available"]  = "—"

# ── 7. Combine adoption + risk ────────────────────────────────────────────────
final = adoption.merge(risk, on=["name", "version"], how="left")

# ── 8. Rename and sort ────────────────────────────────────────────────────────
final = final.rename(columns={
    "name"          : "Library Name",
    "version"       : "Version",
    "Total_CVEs"    : "Total CVEs",
    "Worst_Severity": "Worst Severity",
    "Cnt_Critical"  : "# Critical",
    "Cnt_High"      : "# High",
    "Cnt_Medium"    : "# Medium",
    "Cnt_Low"       : "# Low",
    "Fix_Available" : "Fix Available",
})

# Sort: vulnerable first (by worst severity), then by enterprises descending
sev_sort = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3, "Unknown": 4, "—": 5}
final["_sev_rank"] = final["Worst Severity"].map(sev_sort).fillna(5)
final = final.sort_values(["_sev_rank", "Total Enterprises Exposed"], ascending=[True, False])
final = final.drop(columns=["_sev_rank"]).reset_index(drop=True)

# Final column order
final = final[[
    "Library Name",
    "Version",
    "Total Products Using It",
    "Affected Products",
    "Total Enterprises Exposed",
    "Total CVEs",
    "Worst Severity",
    "# Critical",
    "# High",
    "# Medium",
    "# Low",
    "Fix Available",
]]

# ── 9. Save & summary ────────────────────────────────────────────────────────
final.to_csv(OUTPUT_PATH, index=False)

print(f"\n✅ Done! Saved to: {OUTPUT_PATH}")
print(f"   Total library+version rows : {len(final):,}")
print(f"   Vulnerable libraries        : {(final['Total CVEs'] > 0).sum():,}")
print(f"   Libraries with no CVEs      : {(final['Total CVEs'] == 0).sum():,}")
print(f"   Libraries with Critical CVEs: {(final['# Critical'] > 0).sum():,}")
print(f"   Libraries with a fix        : {(final['Fix Available'] == 'Yes').sum():,}")
print(f"   Libraries partial fix       : {(final['Fix Available'] == 'Partial').sum():,}")
print(f"   Libraries no fix available  : {(final['Fix Available'] == 'No').sum():,}")

print(f"\nTop 20 rows (sorted by severity, then enterprises):")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print(final.head(20).to_string(index=False))

Loading files...
  Libraries rows    : 209,552
  Vulnerability rows: 11,302
  Install rows      : 21,457

Installs unique (vendor, product) pairs: 10,718

⚠  6,385 library rows had no matching installs entry (will count as 0 enterprises).

Distinct (library, CVE) pairs: 7,751
Severity distribution:
severity
High        3371
Medium      2926
Low          813
Critical     641

Building adoption stats...


/var/folders/fj/wtzx880x4v7g0q54zrwpf4gc0000gr/T/ipykernel_92332/2189310441.py:112: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(adoption_agg)


Distinct (library, version) entries: 65,206
Building risk stats...

✅ Done! Saved to: /Users/dmk6603/Documents/swdb_opensource/general_library_risk.csv
   Total library+version rows : 65,206
   Vulnerable libraries        : 2,061
   Libraries with no CVEs      : 63,145
   Libraries with Critical CVEs: 336
   Libraries with a fix        : 1,858
   Libraries partial fix       : 75
   Libraries no fix available  : 127

Top 20 rows (sorted by severity, then enterprises):
   Library Name Version  Total Products Using It                                                                                                                                                                                                                       Affected Products  Total Enterprises Exposed  Total CVEs Worst Severity  # Critical  # High  # Medium  # Low Fix Available
      form-data   2.3.3                       19 Angular Material, AngularJS, Apache Hadoop, Apache Hadoop HDFS, Apache Hadoop MapReduce, Apac

# Count of ONLY Libraries propagation - NOT CONSIDERING VERSION!
## We are not considering only vulnerable libraries, but all libraries

In [18]:
"""
Build: General Library Risk Overview (aggregated by library name, ignoring version)
=====================================================================================
Same logic as general_library_risk.py but groups ALL versions of a library together.

Output columns:
  - Library Name
  - Versions Found              : all distinct versions seen (comma-separated)
  - Total Products Using It     : distinct products that ship any version of this library
  - Affected Products           : comma-separated product names
  - Total Enterprises Exposed   : sum of enterprises across those products (de-duped)
  - Total CVEs                  : distinct CVE count across all versions
  - Worst Severity              : highest severity across all CVEs
  - # Critical / # High / # Medium / # Low
  - Fix Available               : Yes / Partial / No / —

Merge keys:
  libraries → installs : vendor == VendorName  AND  product == Product
  libraries → vulns    : project + name + version == project + pkg_name + pkg_version
"""

import pandas as pd

# ── 0. File paths ─────────────────────────────────────────────────────────────
LIBRARIES_PATH = "/Users/dmk6603/Documents/swdb_opensource/7-vulnerable_libraries/libraries_with_vulns.csv"
VULNS_PATH     = "/Users/dmk6603/Documents/swdb_opensource/5.1-grype_vulnerabilities/vulnerabilities.csv"
INSTALLS_PATH  = "/Users/dmk6603/Documents/swdb_opensource/1-indentify_open_source/data/swdb_universe_installs.csv"
OUTPUT_PATH    = "/Users/dmk6603/Documents/swdb_opensource/general_library_risk_by_name.csv"

# ── 1. Load ───────────────────────────────────────────────────────────────────
print("Loading files...")
libs     = pd.read_csv(LIBRARIES_PATH)
vulns    = pd.read_csv(VULNS_PATH)
installs = pd.read_csv(INSTALLS_PATH)

for df in [libs, vulns, installs]:
    df.columns = df.columns.str.strip()

print(f"  Libraries rows    : {len(libs):,}")
print(f"  Vulnerability rows: {len(vulns):,}")
print(f"  Install rows      : {len(installs):,}")

# ── 2. Clean installs: parse Total Enterprises, de-duplicate ─────────────────
installs["Total Enterprises"] = (
    installs["Total Enterprises"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype(int)
)

installs_dedup = (
    installs
    .groupby(["VendorName", "Product"], as_index=False)["Total Enterprises"]
    .max()
)
print(f"\nInstalls unique (vendor, product) pairs: {len(installs_dedup):,}")

# ── 3. Merge ALL libraries → installs ────────────────────────────────────────
libs_with_ent = libs.merge(
    installs_dedup,
    left_on=["vendor", "product"],
    right_on=["VendorName", "Product"],
    how="left"
)

unmatched = libs_with_ent["Total Enterprises"].isna().sum()
if unmatched > 0:
    print(f"\n⚠  {unmatched:,} library rows had no matching installs entry (will count as 0 enterprises).")

libs_with_ent["Total Enterprises"] = libs_with_ent["Total Enterprises"].fillna(0).astype(int)

# ── 4. Prepare vulnerabilities ────────────────────────────────────────────────
vulns_clean = vulns[["project", "pkg_name", "pkg_version", "cve", "severity", "fix_state"]].copy()
vulns_clean["cve"]      = vulns_clean["cve"].fillna("N/A")
vulns_clean["severity"] = vulns_clean["severity"].fillna("Unknown")
vulns_clean = vulns_clean.drop_duplicates(subset=["project", "pkg_name", "pkg_version", "cve"])

# ── 5. Adoption stats — grouped by library NAME only ─────────────────────────
print("\nBuilding adoption stats (by name only)...")

def adoption_agg(group):
    # De-dup at (vendor, product) level so a library in multiple projects
    # of the same product is counted once
    product_level = (
        group[["vendor", "product", "Total Enterprises"]]
        .drop_duplicates(subset=["vendor", "product"])
    )
    versions = ", ".join(sorted(group["version"].dropna().astype(str).unique()))
    return pd.Series({
        "Versions Found"            : versions,
        "Total Products Using It"   : product_level["product"].nunique(),
        "Affected Products"         : ", ".join(sorted(product_level["product"].dropna().unique())),
        "Total Enterprises Exposed" : product_level["Total Enterprises"].sum(),
    })

adoption = (
    libs_with_ent
    .groupby("name", dropna=False)
    .apply(adoption_agg)
    .reset_index()
)

print(f"Distinct library names: {len(adoption):,}")

# ── 6. Risk stats — grouped by library NAME only ──────────────────────────────
# Join libs → vulns on (project, name, version) then group by name
libs_base = libs[["project", "name", "version"]].drop_duplicates()

libs_with_vulns = libs_base.merge(
    vulns_clean,
    left_on=["project", "name", "version"],
    right_on=["project", "pkg_name", "pkg_version"],
    how="left"
)

SEV_ORDER = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3, "Unknown": 4}

def worst_severity(severities):
    severities = severities.dropna()
    if len(severities) == 0:
        return "—"
    ranked = severities.map(lambda s: SEV_ORDER.get(s, 99))
    return severities.iloc[ranked.argmin()]

def fix_status(fix_states):
    fix_states = fix_states.dropna()
    if len(fix_states) == 0:
        return "—"
    has_fix    = (fix_states.str.lower() == "fixed").sum()
    has_no_fix = (fix_states.str.lower() == "not-fixed").sum()
    if has_fix > 0 and has_no_fix > 0:
        return "Partial"
    elif has_fix > 0:
        return "Yes"
    else:
        return "No"

print("Building risk stats (by name only)...")

risk = (
    libs_with_vulns
    .groupby("name", dropna=False)
    .agg(
        Total_CVEs     = ("cve",       lambda x: x.dropna()[x.dropna() != "N/A"].nunique()),
        Worst_Severity = ("severity",  worst_severity),
        Cnt_Critical   = ("severity",  lambda x: (x == "Critical").sum()),
        Cnt_High       = ("severity",  lambda x: (x == "High").sum()),
        Cnt_Medium     = ("severity",  lambda x: (x == "Medium").sum()),
        Cnt_Low        = ("severity",  lambda x: (x == "Low").sum()),
        Fix_Available  = ("fix_state", fix_status),
    )
    .reset_index()
)

risk.loc[risk["Total_CVEs"] == 0, "Worst_Severity"] = "—"
risk.loc[risk["Total_CVEs"] == 0, "Fix_Available"]  = "—"

# ── 7. Combine adoption + risk ────────────────────────────────────────────────
final = adoption.merge(risk, on="name", how="left")

# ── 8. Rename and sort ────────────────────────────────────────────────────────
final = final.rename(columns={
    "name"          : "Library Name",
    "Total_CVEs"    : "Total CVEs",
    "Worst_Severity": "Worst Severity",
    "Cnt_Critical"  : "# Critical",
    "Cnt_High"      : "# High",
    "Cnt_Medium"    : "# Medium",
    "Cnt_Low"       : "# Low",
    "Fix_Available" : "Fix Available",
})

sev_sort = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3, "Unknown": 4, "—": 5}
final["_sev_rank"] = final["Worst Severity"].map(sev_sort).fillna(5)
final = final.sort_values(["_sev_rank", "Total Enterprises Exposed"], ascending=[True, False])
final = final.drop(columns=["_sev_rank"]).reset_index(drop=True)

final = final[[
    "Library Name",
    "Versions Found",
    "Total Products Using It",
    "Affected Products",
    "Total Enterprises Exposed",
    "Total CVEs",
    "Worst Severity",
    "# Critical",
    "# High",
    "# Medium",
    "# Low",
    "Fix Available",
]]

# ── 9. Save & summary ────────────────────────────────────────────────────────
final.to_csv(OUTPUT_PATH, index=False)

print(f"\n✅ Done! Saved to: {OUTPUT_PATH}")
print(f"   Total library names         : {len(final):,}")
print(f"   Vulnerable libraries        : {(final['Total CVEs'] > 0).sum():,}")
print(f"   Libraries with no CVEs      : {(final['Total CVEs'] == 0).sum():,}")
print(f"   Libraries with Critical CVEs: {(final['# Critical'] > 0).sum():,}")
print(f"   Fix Available - Yes         : {(final['Fix Available'] == 'Yes').sum():,}")
print(f"   Fix Available - Partial     : {(final['Fix Available'] == 'Partial').sum():,}")
print(f"   Fix Available - No          : {(final['Fix Available'] == 'No').sum():,}")

print(f"\nTop 20 rows:")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print(final.head(20).to_string(index=False))

Loading files...
  Libraries rows    : 209,552
  Vulnerability rows: 11,302
  Install rows      : 21,457

Installs unique (vendor, product) pairs: 10,718

⚠  6,385 library rows had no matching installs entry (will count as 0 enterprises).

Building adoption stats (by name only)...


/var/folders/fj/wtzx880x4v7g0q54zrwpf4gc0000gr/T/ipykernel_92332/665825561.py:102: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(adoption_agg)


Distinct library names: 37,940
Building risk stats (by name only)...

✅ Done! Saved to: /Users/dmk6603/Documents/swdb_opensource/general_library_risk_by_name.csv
   Total library names         : 37,940
   Vulnerable libraries        : 627
   Libraries with no CVEs      : 37,313
   Libraries with Critical CVEs: 144
   Fix Available - Yes         : 542
   Fix Available - Partial     : 34
   Fix Available - No          : 51

Top 20 rows:
   Library Name                                                                                                                                                                                                                                                                                                                                            Versions Found  Total Products Using It                                                                                                                                                                               